# Colab Inference - w11wo Indonesian POS Tagger

Notebook ini khusus untuk:
1. Load model `w11wo/indonesian-roberta-base-posp-tagger`
2. Predict 100 pantun awal
3. Evaluasi ke mini-gold (jika tersedia)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import subprocess
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive')
PROJECT_PARENT = DRIVE_ROOT / 'POSTagPantun'
REPO_URL = 'https://github.com/NazirArifin/pantunify.git'
REPO_DIR = PROJECT_PARENT / 'pantunify'
PROJECT_PARENT.mkdir(parents=True, exist_ok=True)

if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
else:
    print('Repo already exists, skip clone.')

%cd {REPO_DIR}
!git --no-pager status -sb


In [ ]:
!python -V
!pip -q install --upgrade pip
!pip -q install torch transformers pandas scikit-learn


In [ ]:
import json
import re
from pathlib import Path
from typing import List

import numpy as np
import pandas as pd
import torch
from transformers import AutoModelForTokenClassification, AutoTokenizer

MODEL_NAME = 'w11wo/indonesian-roberta-base-posp-tagger'
LIMIT_PANTUN = 100
MAX_LENGTH = 128

DATA_DIR = REPO_DIR / 'data' / 'article5'
PANTUN_CSV = DATA_DIR / 'pantun_gaul.csv'
MINI_GOLD_PATH = DATA_DIR / 'processed_crf' / 'mini_gold_template_400.csv'

ARTIFACT_ROOT = PROJECT_PARENT / 'artifacts_w11wo_pos'
OUT_DIR = ARTIFACT_ROOT / MODEL_NAME.replace('/', '__')
OUT_DIR.mkdir(parents=True, exist_ok=True)

print('MODEL:', MODEL_NAME)
print('OUTPUT DIR:', OUT_DIR)
print('PANTUN CSV:', PANTUN_CSV, '->', 'OK' if PANTUN_CSV.exists() else 'MISSING')
print('MINI GOLD :', MINI_GOLD_PATH, '->', 'OK' if MINI_GOLD_PATH.exists() else 'MISSING')


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForTokenClassification.from_pretrained(MODEL_NAME)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
model.eval()

id2label = model.config.id2label
print('device:', device)
print('num_labels:', model.config.num_labels)
print('sample labels:', [id2label[i] for i in range(min(10, len(id2label)))])


In [ ]:
TOKEN_RE = re.compile(r"\w+|[^\w\s]", flags=re.UNICODE)

# Map tagset w11wo (POSP) -> UD coarse tags agar evaluasi apple-to-apple.
W11WO_TO_UD = {
    'ADJ': 'ADJ',
    'ADK': 'ADV',
    'ADV': 'ADV',
    'CSN': 'SCONJ',
    'KUA': 'NUM',
    'NEG': 'PART',
    'NNO': 'NOUN',
    'NNP': 'PROPN',
    'PAR': 'PART',
    'PPO': 'PRON',
    'PRN': 'PRON',
    'SYM': 'SYM',
    'VBE': 'AUX',
    'VBI': 'VERB',
    'VBP': 'VERB',
    'VBT': 'VERB',
}

def tokenize_text(text: str) -> List[str]:
    return TOKEN_RE.findall(text)

def normalize_tag(tag: str) -> str:
    t = str(tag).upper().strip()
    for prefix in ('B-', 'I-', 'S-', 'E-', 'U-', 'L-'):
        if t.startswith(prefix):
            t = t[len(prefix):]
            break
    return W11WO_TO_UD.get(t, t)

def predict_tokens(tokens: List[str]) -> List[str]:
    enc = tokenizer(
        tokens,
        is_split_into_words=True,
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors='pt',
    )
    enc = {k: v.to(device) for k, v in enc.items()}

    with torch.no_grad():
        logits = model(**enc).logits[0].detach().cpu().numpy()
    pred_ids = np.argmax(logits, axis=-1)

    word_ids = tokenizer(tokens, is_split_into_words=True, truncation=True, max_length=MAX_LENGTH).word_ids()
    out = []
    seen = set()
    for tidx, wid in enumerate(word_ids):
        if wid is None or wid in seen:
            continue
        seen.add(wid)
        out.append(normalize_tag(id2label[int(pred_ids[tidx])]))

    if len(out) < len(tokens):
        out += ['NOUN'] * (len(tokens) - len(out))
    return out[:len(tokens)]


In [ ]:
pantun_df = pd.read_csv(PANTUN_CSV)
rows = []

for pantun_index, row in pantun_df.head(LIMIT_PANTUN).iterrows():
    text = str(row.get('text_pantun', '')).replace('\\n', '\n')
    lines = [ln.strip() for ln in text.splitlines() if ln.strip()]
    for line_no, line_text in enumerate(lines, start=1):
        tokens = tokenize_text(line_text)
        if not tokens:
            continue
        pred_tags = predict_tokens(tokens)
        rows.append({
            'id': row.get('id', ''),
            'pantun_index': int(pantun_index) + 1,
            'line_no': line_no,
            'line_text': line_text,
            'token_count': len(tokens),
            'tokens': ' '.join(tokens),
            'pred_tags': ' '.join(pred_tags),
            'token_tag_pairs': ' | '.join(f'{t}/{g}' for t, g in zip(tokens, pred_tags)),
        })

pred_df = pd.DataFrame(rows)
pred_path = OUT_DIR / 'pantun_gaul_pred_100_w11wo.csv'
pred_df.to_csv(pred_path, index=False, encoding='utf-8')

print('[OK] output:', pred_path)
print('[INFO] pantun processed:', min(LIMIT_PANTUN, len(pantun_df)))
print('[INFO] line predictions:', len(pred_df))
display(pred_df.head(10))


In [ ]:
if not MINI_GOLD_PATH.exists():
    print('Mini-gold tidak ditemukan:', MINI_GOLD_PATH)
else:
    mg = pd.read_csv(MINI_GOLD_PATH)
    pred = pd.read_csv(OUT_DIR / 'pantun_gaul_pred_100_w11wo.csv')

    token_rows = []
    for _, row in pred.iterrows():
        toks = str(row['tokens']).split()
        tags = str(row['pred_tags']).split()
        for i, (tok, tag) in enumerate(zip(toks, tags), start=1):
            token_rows.append({
                'pantun_index': int(row['pantun_index']),
                'line_no': int(row['line_no']),
                'token_index': i,
                'token': tok,
                'pred_tag_w11wo': str(tag).upper(),
            })
    pred_tok = pd.DataFrame(token_rows)

    key_cols = ['pantun_index', 'line_no', 'token_index', 'token']
    merged = mg.merge(pred_tok, on=key_cols, how='left')
    merged['pred_tag'] = merged['pred_tag_w11wo'].fillna(merged['pred_tag'])

    eval_csv = OUT_DIR / 'mini_gold_with_w11wo_pred.csv'
    merged.to_csv(eval_csv, index=False, encoding='utf-8')

    metrics_json = OUT_DIR / 'mini_gold_metrics_w11wo.json'
    confusion_csv = OUT_DIR / 'mini_gold_confusion_w11wo.csv'
    !python scripts/article5/evaluate_mini_gold_pos.py \
      --gold-csv {eval_csv} \
      --out-metrics-json {metrics_json} \
      --out-confusion-csv {confusion_csv}

    print('Saved:', metrics_json)
    print('Saved:', confusion_csv)
